In [16]:
"""
Notebook 1: RumourEval 데이터 파싱 + Context 조건 구성
결과물: context_conditions.json
 
데이터 구조 확인 결과:
- Reddit: thread_id/source-tweet/thread_id.json → kind/data/children[0]/data
          thread_id/replies/*.json → data 키
- Twitter: event/thread_id/source-tweet/thread_id.json → Tweet JSON (text, id, ...)
           event/thread_id/replies/*.json → Tweet JSON (text, id, in_reply_to_status_id, ...)
"""
 
# ============================================================
# CELL 1: 임포트
# ============================================================
 
import os
import json
from collections import defaultdict, Counter

In [18]:
# ============================================================
# CELL 2: 데이터 파싱
# ============================================================

def parse_rumoureval(reddit_dirs, twitter_dir, key_files):
    label_map = {}
    for key_file in key_files:
        with open(key_file) as f:
            label_map.update(json.load(f).get('subtaskaenglish', {}))

    records = []

    # Reddit 파싱
    for data_dir, split in reddit_dirs:
        for thread_id in os.listdir(data_dir):
            thread_path = os.path.join(data_dir, thread_id)
            if not os.path.isdir(thread_path):
                continue

            src_dir = os.path.join(thread_path, 'source-tweet')
            if not os.path.exists(src_dir):
                continue
            src_files = os.listdir(src_dir)
            if not src_files:
                continue

            with open(os.path.join(src_dir, src_files[0])) as f:
                src_data = json.load(f)

            src_body = src_data['data']['children'][0]['data']
            src_text = src_body.get('selftext') or src_body.get('title', '')
            src_id   = str(src_body.get('id', thread_id))

            records.append({
                'reply_id':  src_id,
                'thread_id': thread_id,
                'parent_id': None,
                'depth':     -1,
                'text':      src_text,
                'label':     label_map.get(thread_id),
                'platform':  'reddit',
                'split':     split,
                'is_source': True,
            })

            rep_dir = os.path.join(thread_path, 'replies')
            if not os.path.exists(rep_dir):
                continue

            for rep_file in os.listdir(rep_dir):
                with open(os.path.join(rep_dir, rep_file)) as f:
                    rep_data = json.load(f)
                rep_body = rep_data['data']
                rep_id   = str(rep_body.get('id', rep_file.replace('.json', '')))

                records.append({
                    'reply_id':  rep_id,
                    'thread_id': thread_id,
                    'parent_id': str(rep_body.get('parent_id', '').split('_')[-1]),
                    'depth':     rep_body.get('depth', 0),
                    'text':      rep_body.get('body', ''),
                    'label':     label_map.get(rep_id),
                    'platform':  'reddit',
                    'split':     split,
                    'is_source': False,
                })

    # Twitter 파싱
    for event_id in os.listdir(twitter_dir):
        event_path = os.path.join(twitter_dir, event_id)
        if not os.path.isdir(event_path):
            continue

        for thread_id in os.listdir(event_path):
            thread_path = os.path.join(event_path, thread_id)
            if not os.path.isdir(thread_path):
                continue

            src_dir = os.path.join(thread_path, 'source-tweet')
            if not os.path.exists(src_dir):
                continue
            src_files = os.listdir(src_dir)
            if not src_files:
                continue

            with open(os.path.join(src_dir, src_files[0])) as f:
                src_data = json.load(f)

            src_id   = str(src_data.get('id', thread_id))
            src_text = src_data.get('text', '')
            split    = 'dev' if src_id in label_map else 'train'

            records.append({
                'reply_id':  src_id,
                'thread_id': thread_id,
                'parent_id': None,
                'depth':     -1,
                'text':      src_text,
                'label':     label_map.get(src_id),
                'platform':  'twitter',
                'split':     split,
                'is_source': True,
            })

            rep_dir = os.path.join(thread_path, 'replies')
            if not os.path.exists(rep_dir):
                continue

            for rep_file in os.listdir(rep_dir):
                with open(os.path.join(rep_dir, rep_file)) as f:
                    rep_data = json.load(f)

                rep_id    = str(rep_data.get('id', rep_file.replace('.json', '')))
                parent_id = str(rep_data.get('in_reply_to_status_id', ''))

                records.append({
                    'reply_id':  rep_id,
                    'thread_id': thread_id,
                    'parent_id': parent_id,
                    'depth':     0,
                    'text':      rep_data.get('text', ''),
                    'label':     label_map.get(rep_id),
                    'platform':  'twitter',
                    'split':     split,
                    'is_source': False,
                })

    return records


records = parse_rumoureval(
    reddit_dirs=[
        ('reddit-dev-data',      'dev'),
        ('reddit-training-data', 'train'),
    ],
    twitter_dir='twitter-english',
    key_files=['dev-key.json', 'train-key.json']
)

print(f"전체 레코드 수: {len(records)}")
print(f"reddit : {sum(1 for r in records if r['platform']=='reddit')}")
print(f"twitter: {sum(1 for r in records if r['platform']=='twitter')}")
print("\n=== 레이블 분포 (source 제외) ===")
replies = [r for r in records if not r['is_source'] and r['label']]
print(Counter(r['label'] for r in replies))

전체 레코드 수: 6702
reddit : 1134
twitter: 5568

=== 레이블 분포 (source 제외) ===
Counter({'comment': 4689, 'support': 715, 'query': 487, 'deny': 446})


In [22]:
import os
save_path = os.path.expanduser('~/parsed_rumoureval.json')
with open(save_path, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"저장 완료: {save_path} ({len(records)}개)")

저장 완료: /home/ubuntu/parsed_rumoureval.json (6702개)


In [4]:
# ============================================================
# CELL 4: thread 인덱싱
# ============================================================

threads      = defaultdict(list)
for r in records:
    threads[r['thread_id']].append(r)

id_to_record = {r['reply_id']: r for r in records}

print(f"스레드 수: {len(threads)}")

In [5]:
 
# ============================================================
# CELL 5: Context 조건 구성 함수
# ============================================================
 
def build_conditions(target, thread_records, id_to_record):
    src = next((r for r in thread_records if r['is_source']), None)
    if not src or not target['label']:
        return None
 
    src_text  = src['text']
    tgt_text  = target['text']
    tgt_label = target['label']
    parent    = id_to_record.get(target['parent_id'])
 
    # ----------------------------------------------------------
    # 1. reply_only (NC)
    # ----------------------------------------------------------
    reply_only = tgt_text
 
    # ----------------------------------------------------------
    # 2. useful (UC)
    #    source + direct parent(있으면) + target
    # ----------------------------------------------------------
    if parent and not parent['is_source']:
        useful = f"[Source] {src_text}\n[Context] {parent['text']}\n[Target] {tgt_text}"
    else:
        useful = f"[Source] {src_text}\n[Target] {tgt_text}"
 
    # ----------------------------------------------------------
    # 3. irrelevant (TI)
    #    같은 thread의 comment + stance keyword 없음
    #    → target과 Jaccard similarity 가장 낮은 것 선택
    # ----------------------------------------------------------
    ti_candidates = [
        r for r in thread_records
        if r['label'] == 'comment'
        and r['reply_id'] != target['reply_id']
        and r['reply_id'] != target['parent_id']
        and not r['is_source']
        and not has_stance_keyword(r['text'])
    ]
 
    if ti_candidates:
        ti_ctx_record = min(
            ti_candidates,
            key=lambda r: jaccard_similarity(r['text'], tgt_text)
        )
        irrelevant = f"[Source] {src_text}\n[Context] {ti_ctx_record['text']}\n[Target] {tgt_text}"
        valid_ti   = True
    else:
        irrelevant = f"[Source] {src_text}\n[Target] {tgt_text}"
        valid_ti   = False
 
    # ----------------------------------------------------------
    # 4. conflicting (CC)
    #    반대 stance label reply → stance keyword 가장 많은 것 선택
    # ----------------------------------------------------------
    conflict_pref = {
        'support': ['deny'],
        'deny':    ['support'],
        'query':   ['support', 'deny'],
        'comment': ['deny', 'support'],
    }
 
    cc_candidates = [
        r for r in thread_records
        if r['label'] in conflict_pref.get(tgt_label, [])
        and r['label'] != tgt_label
        and r['reply_id'] != target['reply_id']
        and not r['is_source']
        and len(r['text'].split()) >= 5
    ]
 
    if cc_candidates:
        cc_ctx_record = max(
            cc_candidates,
            key=lambda r: sum(1 for kw in STANCE_KEYWORDS if kw in r['text'].lower())
        )
        conflicting = f"[Source] {src_text}\n[Context] {cc_ctx_record['text']}\n[Target] {tgt_text}"
        valid_cc    = True
    else:
        conflicting = None
        valid_cc    = False
 
    # ----------------------------------------------------------
    # 5. mixed
    #    useful + conflicting 혼합
    # ----------------------------------------------------------
    if valid_cc:
        if parent and not parent['is_source']:
            mixed = (f"[Source] {src_text}\n"
                     f"[Context] {parent['text']}\n"
                     f"[Misleading] {cc_ctx_record['text']}\n"
                     f"[Target] {tgt_text}")
        else:
            mixed = (f"[Source] {src_text}\n"
                     f"[Misleading] {cc_ctx_record['text']}\n"
                     f"[Target] {tgt_text}")
    else:
        mixed = None
 
 
    return {
        'reply_id':   target['reply_id'],
        'thread_id':  target['thread_id'],
        'label':      tgt_label,
        'platform':   target['platform'],
        'split':      target['split'],
        'reply_only': reply_only,
        'useful':     useful,
        'irrelevant': irrelevant,
        'conflicting':conflicting,
        'mixed':      mixed,
        'valid_ti':   valid_ti,
        'valid_cc':   valid_cc,
    }

In [6]:
 
# ============================================================
# CELL 6: 전체 데이터셋 구성
# ============================================================
 
dataset = []
for target in records:
    if target['is_source'] or not target['label']:
        continue
    thread_records = threads[target['thread_id']]
    result = build_conditions(target, thread_records, id_to_record)
    if result:
        dataset.append(result)
 
print(f"완성된 샘플 수: {len(dataset)}")
print(f"valid_cc 비율: {sum(1 for d in dataset if d['valid_cc']) / len(dataset) * 100:.1f}%")
print(f"valid_ti 비율: {sum(1 for d in dataset if d['valid_ti']) / len(dataset) * 100:.1f}%")
 
print("\n=== 레이블별 valid_cc 비율 ===")
for label in ['support', 'deny', 'query', 'comment']:
    subset = [d for d in dataset if d['label'] == label]
    valid  = [d for d in subset  if d['valid_cc']]
    print(f"  {label:10s}: {len(valid)}/{len(subset)} ({len(valid)/len(subset)*100:.1f}%)")
 
print("\n=== platform별 분포 ===")
for platform in ['twitter', 'reddit']:
    subset = [d for d in dataset if d['platform'] == platform]
    print(f"  {platform:10s}: {len(subset)}개")
 
print("\n=== irrelevant fallback 발생 수 ===")
fallback = [d for d in dataset if not d['valid_ti']]
print(f"  총 {len(fallback)}개 → source+target만 사용")

완성된 샘플 수: 6337
valid_cc 비율: 82.7%
valid_ti 비율: 96.4%

=== 레이블별 valid_cc 비율 ===
  support   : 436/715 (61.0%)
  deny      : 353/446 (79.1%)
  query     : 427/487 (87.7%)
  comment   : 4022/4689 (85.8%)

=== platform별 분포 ===
  twitter   : 5243개
  reddit    : 1094개

=== irrelevant fallback 발생 수 ===
  총 226개 → source+target만 사용


In [7]:
# ============================================================
# CELL 7: 샘플 확인
# ============================================================
 
import pprint
print("=== 샘플 1개 확인 ===")
pprint.pprint(dataset[0])

=== 샘플 1개 확인 ===
{'conflicting': None,
 'irrelevant': '[Source] Fukushima spewing equivalent of 112 Hiroshima-type '
               'nuclear bombs worth of radiation every hour, of every day. '
               '(This canNOT be true...)\n'
               '[Context] The iodine has an 8 day halflife the caesium 30 '
               'years\n'
               '[Target] \n'
               'Fukushima is still in the clean-up process. There are still '
               'issues with radiation leakages - now whilst *any* radiation '
               'leak is considered serious, technically speaking it is only '
               'affecting a small area, and should only concern e.g. locals '
               'and fishermen\n'
               '\n'
               'The article in question is highly dubious and not to be '
               'trusted.\n'
               '\n'
               'EPA spokesmen and US experts have said there is nothing to '
               'worry about from a US point of view. \n'
           

In [8]:
import json

with open('context_conditions.json', 'w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

print('저장 완료!')
print(f'파일 확인: context_conditions.json')

저장 완료!
파일 확인: context_conditions.json


In [9]:
# ============================================================
# CELL 8: 저장
# ============================================================
 
with open('context_conditions.json', 'w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)
 
print("저장 완료: context_conditions.json")

저장 완료: context_conditions.json


In [10]:
with open('context_conditions.json', 'w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

print('저장 완료!')

저장 완료!


In [11]:
"""
Notebook 1 v3: RumourEval 데이터 파싱 + Context 조건 구성
수정사항: dev/train split 판별 로직 수정 (reply ID 기준으로 판별)
결과물: context_conditions.json
"""

# ============================================================
# CELL 1: 임포트
# ============================================================

import os
import json
from collections import defaultdict, Counter


# ============================================================
# CELL 2: 레이블 맵 + split 판별 함수 로드
# ============================================================

with open('dev-key.json') as f:
    dev_label_map = json.load(f).get('subtaskaenglish', {})

with open('train-key.json') as f:
    train_label_map = json.load(f).get('subtaskaenglish', {})

# 전체 label_map (합치기)
label_map = {**dev_label_map, **train_label_map}

print(f"dev 레이블 수  : {len(dev_label_map)}")
print(f"train 레이블 수: {len(train_label_map)}")
print(f"겹치는 ID      : {len(set(dev_label_map) & set(train_label_map))}")


def get_split(reply_id: str) -> str:
    """reply_id 기준으로 dev/train/unknown 반환"""
    if reply_id in dev_label_map:
        return 'dev'
    elif reply_id in train_label_map:
        return 'train'
    else:
        return 'unknown'


# ============================================================
# CELL 3: 데이터 파싱
# ============================================================

def parse_rumoureval(reddit_dirs, twitter_dir):
    records = []

    # ----------------------------------------------------------
    # Reddit 파싱
    # 구조: data_dir/thread_id/source-tweet/thread_id.json
    #        data_dir/thread_id/replies/*.json
    # ----------------------------------------------------------
    for data_dir, _ in reddit_dirs:
        for thread_id in os.listdir(data_dir):
            thread_path = os.path.join(data_dir, thread_id)
            if not os.path.isdir(thread_path):
                continue

            # source-tweet
            src_dir = os.path.join(thread_path, 'source-tweet')
            if not os.path.exists(src_dir):
                continue
            src_files = os.listdir(src_dir)
            if not src_files:
                continue

            with open(os.path.join(src_dir, src_files[0])) as f:
                src_data = json.load(f)

            src_body = src_data['data']['children'][0]['data']
            src_text = src_body.get('selftext') or src_body.get('title', '')
            src_id   = str(src_body.get('id', thread_id))

            records.append({
                'reply_id':  src_id,
                'thread_id': thread_id,
                'parent_id': None,
                'depth':     -1,
                'text':      src_text,
                'label':     label_map.get(thread_id),
                'platform':  'reddit',
                'split':     get_split(thread_id),
                'is_source': True,
            })

            # replies
            rep_dir = os.path.join(thread_path, 'replies')
            if not os.path.exists(rep_dir):
                continue

            for rep_file in os.listdir(rep_dir):
                with open(os.path.join(rep_dir, rep_file)) as f:
                    rep_data = json.load(f)
                rep_body = rep_data['data']
                rep_id   = str(rep_body.get('id', rep_file.replace('.json', '')))

                records.append({
                    'reply_id':  rep_id,
                    'thread_id': thread_id,
                    'parent_id': str(rep_body.get('parent_id', '').split('_')[-1]),
                    'depth':     rep_body.get('depth', 0),
                    'text':      rep_body.get('body', ''),
                    'label':     label_map.get(rep_id),
                    'platform':  'reddit',
                    'split':     get_split(rep_id),
                    'is_source': False,
                })

    # ----------------------------------------------------------
    # Twitter 파싱
    # 구조: twitter_dir/event_id/thread_id/source-tweet/thread_id.json
    #        twitter_dir/event_id/thread_id/replies/*.json
    # JSON 키: text, id, in_reply_to_status_id
    # ※ depth 정보 없음 → 0으로 통일
    # ----------------------------------------------------------
    for event_id in os.listdir(twitter_dir):
        event_path = os.path.join(twitter_dir, event_id)
        if not os.path.isdir(event_path):
            continue

        for thread_id in os.listdir(event_path):
            thread_path = os.path.join(event_path, thread_id)
            if not os.path.isdir(thread_path):
                continue

            # source-tweet
            src_dir = os.path.join(thread_path, 'source-tweet')
            if not os.path.exists(src_dir):
                continue
            src_files = os.listdir(src_dir)
            if not src_files:
                continue

            with open(os.path.join(src_dir, src_files[0])) as f:
                src_data = json.load(f)

            src_id   = str(src_data.get('id', thread_id))
            src_text = src_data.get('text', '')

            records.append({
                'reply_id':  src_id,
                'thread_id': thread_id,
                'parent_id': None,
                'depth':     -1,
                'text':      src_text,
                'label':     label_map.get(src_id),
                'platform':  'twitter',
                'split':     get_split(src_id),
                'is_source': True,
            })

            # replies
            rep_dir = os.path.join(thread_path, 'replies')
            if not os.path.exists(rep_dir):
                continue

            for rep_file in os.listdir(rep_dir):
                with open(os.path.join(rep_dir, rep_file)) as f:
                    rep_data = json.load(f)

                rep_id    = str(rep_data.get('id', rep_file.replace('.json', '')))
                parent_id = str(rep_data.get('in_reply_to_status_id', ''))

                records.append({
                    'reply_id':  rep_id,
                    'thread_id': thread_id,
                    'parent_id': parent_id,
                    'depth':     0,
                    'text':      rep_data.get('text', ''),
                    'label':     label_map.get(rep_id),
                    'platform':  'twitter',
                    'split':     get_split(rep_id),
                    'is_source': False,
                })

    return records


records = parse_rumoureval(
    reddit_dirs=[
        ('reddit-dev-data',      'dev'),
        ('reddit-training-data', 'train'),
    ],
    twitter_dir='twitter-english',
)

print(f"\n전체 레코드 수: {len(records)}")
print(f"레이블 있는 것: {sum(1 for r in records if r['label'])}")
print(f"reddit : {sum(1 for r in records if r['platform']=='reddit')}")
print(f"twitter: {sum(1 for r in records if r['platform']=='twitter')}")

print("\n=== split 분포 ===")
print(Counter(r['split'] for r in records if not r['is_source'] and r['label']))

print("\n=== 레이블 분포 (source 제외, 전체) ===")
replies = [r for r in records if not r['is_source'] and r['label']]
print(Counter(r['label'] for r in replies))

print("\n=== 레이블 분포 (dev만) ===")
dev_replies = [r for r in records if not r['is_source'] and r['label'] and r['split'] == 'dev']
print(Counter(r['label'] for r in dev_replies))


# ============================================================
# CELL 4: thread 인덱싱
# ============================================================

threads      = defaultdict(list)
for r in records:
    threads[r['thread_id']].append(r)

id_to_record = {r['reply_id']: r for r in records}

print(f"스레드 수: {len(threads)}")


# ============================================================
# CELL 5: 유틸리티 함수
# ============================================================

STANCE_KEYWORDS = [
    "confirmed", "confirm", "true", "false", "fake", "real",
    "denied", "deny", "hoax", "verified", "debunked", "proven",
    "official", "breaking", "fact"
]

LD_TEMPLATES = {
    'support': "This has been widely denied and officially debunked by experts.",
    'deny':    "Multiple credible sources have confirmed this to be completely true.",
    'query':   "There is absolutely no doubt about this. It has been verified.",
    'comment': "This claim has been officially confirmed as false by authorities.",
}


def has_stance_keyword(text):
    return any(kw in text.lower() for kw in STANCE_KEYWORDS)


def jaccard_similarity(text_a, text_b):
    """
    단어 집합 기반 Jaccard similarity.
    낮을수록 두 텍스트가 어휘적으로 다름.
    """
    set_a = set(text_a.lower().split())
    set_b = set(text_b.lower().split())
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


# ============================================================
# CELL 6: Context 조건 구성 함수
# ============================================================

def build_conditions(target, thread_records, id_to_record):
    src = next((r for r in thread_records if r['is_source']), None)
    if not src or not target['label']:
        return None

    src_text  = src['text']
    tgt_text  = target['text']
    tgt_label = target['label']
    parent    = id_to_record.get(target['parent_id'])

    # 1. reply_only
    reply_only = tgt_text

    # 2. useful
    if parent and not parent['is_source']:
        useful = f"[Source] {src_text}\n[Context] {parent['text']}\n[Target] {tgt_text}"
    else:
        useful = f"[Source] {src_text}\n[Target] {tgt_text}"

    # 3. irrelevant
    ti_candidates = [
        r for r in thread_records
        if r['label'] == 'comment'
        and r['reply_id'] != target['reply_id']
        and r['reply_id'] != target['parent_id']
        and not r['is_source']
        and not has_stance_keyword(r['text'])
    ]

    if ti_candidates:
        ti_ctx_record = min(
            ti_candidates,
            key=lambda r: jaccard_similarity(r['text'], tgt_text)
        )
        irrelevant = f"[Source] {src_text}\n[Context] {ti_ctx_record['text']}\n[Target] {tgt_text}"
        valid_ti   = True
    else:
        irrelevant = f"[Source] {src_text}\n[Target] {tgt_text}"
        valid_ti   = False

    # 4. conflicting
    conflict_pref = {
        'support': ['deny'],
        'deny':    ['support'],
        'query':   ['support', 'deny'],
        'comment': ['deny', 'support'],
    }

    cc_candidates = [
        r for r in thread_records
        if r['label'] in conflict_pref.get(tgt_label, [])
        and r['label'] != tgt_label
        and r['reply_id'] != target['reply_id']
        and not r['is_source']
        and len(r['text'].split()) >= 5
    ]

    if cc_candidates:
        cc_ctx_record = max(
            cc_candidates,
            key=lambda r: sum(1 for kw in STANCE_KEYWORDS if kw in r['text'].lower())
        )
        conflicting = f"[Source] {src_text}\n[Context] {cc_ctx_record['text']}\n[Target] {tgt_text}"
        valid_cc    = True
    else:
        conflicting = None
        valid_cc    = False

    # 5. mixed
    if valid_cc:
        if parent and not parent['is_source']:
            mixed = (f"[Source] {src_text}\n"
                     f"[Context] {parent['text']}\n"
                     f"[Misleading] {cc_ctx_record['text']}\n"
                     f"[Target] {tgt_text}")
        else:
            mixed = (f"[Source] {src_text}\n"
                     f"[Misleading] {cc_ctx_record['text']}\n"
                     f"[Target] {tgt_text}")
    else:
        mixed = None

    # 6. lexical
    lexical = f"[Context] {LD_TEMPLATES[tgt_label]}\n[Target] {tgt_text}"

    return {
        'reply_id':   target['reply_id'],
        'thread_id':  target['thread_id'],
        'label':      tgt_label,
        'platform':   target['platform'],
        'split':      target['split'],
        'reply_only': reply_only,
        'useful':     useful,
        'irrelevant': irrelevant,
        'conflicting':conflicting,
        'mixed':      mixed,
        'lexical':    lexical,
        'valid_ti':   valid_ti,
        'valid_cc':   valid_cc,
    }


# ============================================================
# CELL 7: 전체 데이터셋 구성
# ============================================================

dataset = []
for target in records:
    if target['is_source'] or not target['label']:
        continue
    thread_records = threads[target['thread_id']]
    result = build_conditions(target, thread_records, id_to_record)
    if result:
        dataset.append(result)

print(f"완성된 샘플 수 (전체): {len(dataset)}")

dev_dataset = [d for d in dataset if d['split'] == 'dev']
print(f"완성된 샘플 수 (dev) : {len(dev_dataset)}")

print(f"\nvalid_cc 비율 (dev): {sum(1 for d in dev_dataset if d['valid_cc']) / len(dev_dataset) * 100:.1f}%")
print(f"valid_ti 비율 (dev): {sum(1 for d in dev_dataset if d['valid_ti']) / len(dev_dataset) * 100:.1f}%")

print("\n=== 레이블별 valid_cc 비율 (dev) ===")
for label in ['support', 'deny', 'query', 'comment']:
    subset = [d for d in dev_dataset if d['label'] == label]
    valid  = [d for d in subset if d['valid_cc']]
    print(f"  {label:10s}: {len(valid)}/{len(subset)} ({len(valid)/len(subset)*100:.1f}%)")

print("\n=== platform별 분포 (dev) ===")
for platform in ['twitter', 'reddit']:
    subset = [d for d in dev_dataset if d['platform'] == platform]
    print(f"  {platform:10s}: {len(subset)}개")

print("\n=== irrelevant fallback 발생 수 (dev) ===")
fallback = [d for d in dev_dataset if not d['valid_ti']]
print(f"  총 {len(fallback)}개 → source+target만 사용")


# ============================================================
# CELL 8: 저장
# ============================================================

with open('context_conditions.json', 'w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

print("\n저장 완료: context_conditions.json")
print(f"전체 {len(dataset)}개 저장 (dev {len(dev_dataset)}개 포함)")

dev 레이블 수  : 1485
train 레이블 수: 5217
겹치는 ID      : 0

전체 레코드 수: 6702
레이블 있는 것: 6702
reddit : 1134
twitter: 5568

=== split 분포 ===
Counter({'train': 4890, 'dev': 1447})

=== 레이블 분포 (source 제외, 전체) ===
Counter({'comment': 4689, 'support': 715, 'query': 487, 'deny': 446})

=== 레이블 분포 (dev만) ===
Counter({'comment': 1181, 'query': 114, 'deny': 79, 'support': 73})
스레드 수: 365
완성된 샘플 수 (전체): 6337
완성된 샘플 수 (dev) : 1447

valid_cc 비율 (dev): 86.0%
valid_ti 비율 (dev): 96.8%

=== 레이블별 valid_cc 비율 (dev) ===
  support   : 46/73 (63.0%)
  deny      : 68/79 (86.1%)
  query     : 99/114 (86.8%)
  comment   : 1031/1181 (87.3%)

=== platform별 분포 (dev) ===
  twitter   : 1021개
  reddit    : 426개

=== irrelevant fallback 발생 수 (dev) ===
  총 47개 → source+target만 사용

저장 완료: context_conditions.json
전체 6337개 저장 (dev 1447개 포함)


In [12]:
# dev-key.json 레이블 분포 직접 확인
with open('dev-key.json') as f:
    dev_key = json.load(f)

from collections import Counter
dev_labels = dev_key.get('subtaskaenglish', {})
print(Counter(dev_labels.values()))

Counter({'comment': 1181, 'query': 120, 'support': 102, 'deny': 82})


In [13]:
# 누락된 샘플이 어떤 것들인지 확인
with open('dev-key.json') as f:
    dev_key = json.load(f)

dev_labels = dev_key.get('subtaskaenglish', {})

# dev_dataset에 있는 reply_id 집합
parsed_ids = {d['reply_id'] for d in dataset if d['split'] == 'dev'}

# dev-key에는 있는데 파싱된 데이터에 없는 것들
missing = {rid: label for rid, label in dev_labels.items() if rid not in parsed_ids}

print(f"누락 총계: {len(missing)}개")
print(f"누락 레이블 분포: {Counter(missing.values())}")

누락 총계: 38개
누락 레이블 분포: Counter({'support': 29, 'query': 6, 'deny': 3})


In [14]:
# 누락된 샘플이 twitter인지 reddit인지 확인
missing_ids = {rid for rid, label in dev_labels.items() if rid not in parsed_ids}

# records에서 찾아보기
found_in_records = [r for r in records if r['reply_id'] in missing_ids]
not_in_records   = missing_ids - {r['reply_id'] for r in records}

print(f"records에는 있는데 dataset에서 빠진 것: {len(found_in_records)}개")
print(f"records 자체에도 없는 것: {len(not_in_records)}개")

if found_in_records:
    print(Counter(r['platform'] for r in found_in_records))

records에는 있는데 dataset에서 빠진 것: 38개
records 자체에도 없는 것: 0개
Counter({'twitter': 28, 'reddit': 10})


In [15]:
# 왜 걸러졌는지 확인
missing_ids = {rid for rid, label in dev_labels.items() if rid not in parsed_ids}

for r in records:
    if r['reply_id'] not in missing_ids:
        continue
    
    # build_conditions에서 걸러지는 조건 체크
    thread_records = threads[r['thread_id']]
    src = next((x for x in thread_records if x['is_source']), None)
    
    print(f"id: {r['reply_id']}")
    print(f"  is_source: {r['is_source']}")
    print(f"  label: {r['label']}")
    print(f"  src 있음: {src is not None}")
    print(f"  text 있음: {bool(r['text'])}")
    print()

id: 1jvbd8
  is_source: True
  label: deny
  src 있음: True
  text 있음: True

id: 4dfdvo
  is_source: True
  label: query
  src 있음: True
  text 있음: True

id: 6hb6b3
  is_source: True
  label: query
  src 있음: True
  text 있음: True

id: 5qzxep
  is_source: True
  label: query
  src 있음: True
  text 있음: True

id: 31xv6u
  is_source: True
  label: support
  src 있음: True
  text 있음: True

id: 83ddtk
  is_source: True
  label: query
  src 있음: True
  text 있음: True

id: 8j9s33
  is_source: True
  label: support
  src 있음: True
  text 있음: True

id: 66yxyf
  is_source: True
  label: query
  src 있음: True
  text 있음: True

id: 934q6t
  is_source: True
  label: support
  src 있음: True
  text 있음: True

id: 7d28gk
  is_source: True
  label: query
  src 있음: True
  text 있음: True

id: 769988636754505729
  is_source: True
  label: support
  src 있음: True
  text 있음: True

id: 763098277986209792
  is_source: True
  label: support
  src 있음: True
  text 있음: True

id: 764927075522260992
  is_source: True
  label: suppo